# Common generalized Littlewood–Richardson coefficients — optimized SageMath notebook

For partitions $\lambda,\mu$ and $0\le k\le\min(|\lambda|,|\mu|)$, this notebook computes

$$S_k^{\lambda,\mu}(\lambda',\mu') = \sum_{|\delta|+|\gamma|=k} N^\lambda_{\delta,\gamma,\lambda'}N^\mu_{\delta,\gamma,\mu'}.$$

The sums are **over ordered pairs** $(\delta,\gamma)$, including the empty partition. The notebook preserves your original multiplicities and produces LaTeX tables. It avoids enumerating all $\lambda',\mu',\delta,\gamma$ combinations by using `lrcalc.coprod` and a reusable bilinear kernel. It requires a **SageMath kernel**, not ordinary Python.

### Mathematical reorganization

By associativity,
$$N^\lambda_{\delta,\gamma,\lambda'} = \sum_{\nu\vdash k}c^\nu_{\delta,\gamma}c^\lambda_{\nu,\lambda'}.$$
Define the symmetric kernel
$$K_k(\nu,\omega)=\sum_{|\delta|+|\gamma|=k}c^\nu_{\delta,\gamma}c^\omega_{\delta,\gamma}.$$
Then
$$\boxed{S_k^{\lambda,\mu}(\lambda',\mu')=\sum_{\nu,\omega\vdash k}c^\lambda_{\nu,\lambda'}K_k(\nu,\omega)c^\mu_{\omega,\mu'}}.$$

`coprod(λ, all=1)` returns the sparse coefficients $c^\lambda_{\nu,\lambda'}$ for all ordered pairs, so we use those entries directly. The kernel depends only on $(\nu,\omega)$ and can be reused across rows and tables.

In [1]:
from collections import defaultdict
from functools import lru_cache
from pathlib import Path
import sage.libs.lrcalc.lrcalc as lrcalc

def part(p):
    p = tuple(int(x) for x in p)
    if any(x <= 0 for x in p) or any(a < b for a, b in zip(p, p[1:])):
        raise ValueError(f"Invalid partition: {p}")
    return p

def degree(p): return sum(p)

@lru_cache(maxsize=None)
def lr(outer, left, right):
    outer, left, right = part(outer), part(left), part(right)
    if degree(outer) != degree(left) + degree(right): return 0
    return int(lrcalc.lrcoef_unsafe(outer, left, right))

@lru_cache(maxsize=None)
def coproduct(p):
    """Sparse map (left, right) -> c^p_{left,right}; all *ordered* pairs."""
    p = part(p)
    return {(part(left), part(right)): int(coef)
            for (left, right), coef in lrcalc.coprod(p, all=1).items()}

@lru_cache(maxsize=None)
def triple_lr(left, middle, right, outer):
    """N^outer_{left,middle,right} using one cached Schur product."""
    left, middle, right, outer = map(part, (left, middle, right, outer))
    if degree(left) + degree(middle) + degree(right) != degree(outer): return 0
    return sum(int(c) * lr(outer, part(nu), right)
               for nu, c in lrcalc.mult(left, middle).items())

print('SageMath LR routines ready')

SageMath LR routines ready


## Kernel and row construction

For fixed $k$, the only potentially nonzero terms have $|\lambda'|=|\lambda|-k$ and $|\mu'|=|\mu|-k$. We build sparse maps indexed by the right-hand partition, rather than checking all partitions of those degrees.

All coproduct multiplicities remain exact integers; no approximation is introduced.

In [2]:
@lru_cache(maxsize=None)
def coproduct_by_right(p, k):
    """Map p_right -> {p_left: c^p_{p_left,p_right}} for |p_left|=k."""
    grouped = defaultdict(dict)
    for (left, right), coeff in coproduct(part(p)).items():
        if degree(left) == k:
            grouped[right][left] = coeff
    return dict(grouped)

@lru_cache(maxsize=None)
def kernel(nu, omega):
    """Sum_{d,g} c^nu_{d,g} c^omega_{d,g}, with ordered (d,g)."""
    nu, omega = part(nu), part(omega)
    if degree(nu) != degree(omega): return 0
    # dot product of two sparse coproduct vectors
    a, b = coproduct(nu), coproduct(omega)
    if len(a) > len(b): a, b = b, a
    return sum(value * b.get(pair, 0) for pair, value in a.items())

@lru_cache(maxsize=None)
def row_data(k, lam, mu):
    """Return {(lam_prime, mu_prime): multiplicity} for one k."""
    lam, mu = part(lam), part(mu)
    if not 0 <= k <= min(degree(lam), degree(mu)): return {}
    A = coproduct_by_right(lam, k)
    B = coproduct_by_right(mu, k)
    if not A or not B: return {}
    all_nu = sorted({nu for terms in A.values() for nu in terms})
    all_omega = sorted({omega for terms in B.values() for omega in terms})
    K = {(nu, omega): kernel(nu, omega) for nu in all_nu for omega in all_omega}
    result = {}
    for lp, left_terms in A.items():
        for mp, right_terms in B.items():
            total = sum(ca * cb * K[(nu, omega)]
                        for nu, ca in left_terms.items()
                        for omega, cb in right_terms.items())
            if total:
                result[(lp, mp)] = total
    return result


def max_degree(lam, mu):
    """Largest k having a nonzero row; not necessarily every smaller row is nonzero."""
    return max((k for k in range(min(degree(lam), degree(mu)) + 1)
                if row_data(k, part(lam), part(mu))), default=0)

print('Kernel and row functions ready')

Kernel and row functions ready


## Table formatting and compilable LaTeX output

`table_tex()` still returns a single table fragment. Both `write_table()` and `generate_tables()` now write **complete standalone LaTeX documents by default**, using the supplied article preamble, author, title, `\maketitle`, `\renewcommand{\arraystretch}{1.6}`, and the matching `table[H]` environment.

Set `standalone=False` to export fragments for an existing LaTeX document. The LR computations and row selection are unchanged.


In [3]:
# Standalone LaTeX document configuration. Change these values as needed.
LATEX_AUTHOR = "Pablo Santos"
LATEX_TITLE = None  # Auto-generated as "DegN partitions" by generate_tables().

LATEX_PREAMBLE = r"""\documentclass[a4paper,12pt]{article}
\usepackage[margin=1in]{geometry}
\usepackage{amsmath,amssymb,amsthm}
\usepackage{bm}
\usepackage{enumitem}
\usepackage{listings}
\usepackage{xcolor}
\usepackage{hyperref}
\usepackage{cleveref}
\usepackage{booktabs}
\usepackage{array}
\usepackage{graphicx}
\usepackage{float}
\usepackage{caption}
\usepackage{subcaption}
\usepackage{authblk}
\usepackage{tabularx}
"""

def latex_part(p):
    p = part(p)
    return r"\varnothing" if not p else "(" + ",".join(map(str, p)) + ")"

def direct_sum(k, lam, mu):
    data = row_data(k, part(lam), part(mu))
    terms = []
    for (lp, mp), multiplicity in sorted(data.items(), reverse=True):
        prefix = "" if multiplicity == 1 else str(multiplicity)
        terms.append(prefix + r"V^{" + latex_part(lp) + "," + latex_part(mp) + "}")
    return r" \oplus ".join(terms) if terms else "0"

def table_tex(lam, mu, include_zeros=False):
    """One LaTeX table (fragment); coefficients and summation unchanged."""
    lam, mu = part(lam), part(mu)
    kmax = min(degree(lam), degree(mu))
    rows = []
    for k in range(kmax, -1, -1):
        expression = direct_sum(k, lam, mu)
        if include_zeros or expression != "0":
            rows.append(rf"$k = {k}: \quad {expression}$ \\ \hline")
    title = rf"Table for $\lambda = {latex_part(lam)}$, $\mu = {latex_part(mu)}$:"
    return "\n".join([
        r"\begin{table}[H]", r"\centering",
        rf"\caption{{{title}}}",
        r"\begin{tabular}{|>{\centering\arraybackslash}p{15cm}|}",
        r"\hline", *rows,
        r"\end{tabular}", r"\end{table}", ""
    ])

def document_start(title, author=LATEX_AUTHOR):
    """Complete standalone LaTeX header, including title and document start."""
    # Escape only text for LaTeX metadata, not math expressions in table rows.
    title = str(title).replace("&", r"\&").replace("%", r"\%")
    author = str(author).replace("&", r"\&").replace("%", r"\%")
    return (LATEX_PREAMBLE + "\n" +
            rf"\title{{{title}}}" + "\n" +
            rf"\author{{{author}}}" + "\n" +
            r"\date{\today}" + "\n\n" +
            r"\begin{document}" + "\n" +
            r"\maketitle" + "\n" +
            r"\renewcommand{\arraystretch}{1.6}" + "\n\n")

def document_end():
    return "\n\\end{document}\n"

def write_table(lam, mu, filename, append=False, include_zeros=False,
                standalone=True, title=None, author=LATEX_AUTHOR):
    """Write a standalone, compilable .tex file by default.

    Set standalone=False for a table fragment. If append=True and standalone=True,
    insert the table before an existing \\end{document} when present.
    """
    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    body = table_tex(lam, mu, include_zeros=include_zeros)
    title = title or LATEX_TITLE or f"Deg{max(degree(lam), degree(mu))} partitions"
    if append and path.exists():
        old = path.read_text(encoding="utf-8")
        if standalone:
            marker = r"\end{document}"
            if marker not in old:
                raise ValueError("Existing file lacks \\end{document}; cannot append safely")
            old = old.rsplit(marker, 1)[0]
            path.write_text(old + body + document_end(), encoding="utf-8")
        else:
            with path.open("a", encoding="utf-8") as f:
                f.write(body + "\n")
    else:
        text = document_start(title, author) + body + document_end() if standalone else body
        path.write_text(text, encoding="utf-8")
    return path

# Example: export one complete, compilable .tex document
# write_table((), (), 'tex_tables/single_table.tex')


## Small-degree verification against the original definition

These checks evaluate the defining sum explicitly for small partitions and compare it with the factored formula. They also check the example from your notebook. Validation is important because a misplaced symmetry factor of 2 would change every multiplicity.

In [4]:
@lru_cache(maxsize=None)
def partitions_of(n, maxpart=None):
    n = int(n)
    if n == 0: return ((),)
    cap = n if maxpart is None else min(n, maxpart)
    return tuple((first,) + rest
                 for first in range(cap, 0, -1)
                 for rest in partitions_of(n - first, first))

def reference_coefficient(k, lam, lp, mu, mp):
    """Direct implementation of the original defining sum, for testing only."""
    lam, lp, mu, mp = map(part, (lam, lp, mu, mp))
    if degree(lam) - degree(lp) != k or degree(mu) - degree(mp) != k: return 0
    return sum(triple_lr(d,g,lp,lam)*triple_lr(d,g,mp,mu)
               for i in range(k+1)
               for d in partitions_of(i)
               for g in partitions_of(k-i))

assert triple_lr((1,1), (1,1), (1,1), (3,3)) == 1
assert row_data(0, (2,1), (2,)) == {((2,1),(2,)): 1}
assert row_data(12, (5,3,2,2), (4,4,4)).get(((),())) == 18

for lam, mu in [((2,1),(2,1)), ((3,1),(2,2)), ((2,2),(3,1))]:
    for k in range(min(degree(lam), degree(mu)) + 1):
        fast = row_data(k, lam, mu)
        for lp in partitions_of(degree(lam)-k):
            for mp in partitions_of(degree(mu)-k):
                slow = reference_coefficient(k, lam, lp, mu, mp)
                assert fast.get((lp, mp), 0) == slow, (lam,mu,k,lp,mp,slow)
print('All small-degree checks passed')

All small-degree checks passed


## Batch generation: standalone PDF-ready LaTeX source

`generate_tables(4, 4, "tex_tables/deg4_partitions.tex")` writes one `article` document with **all tables**, the provided package imports, title `Deg4 partitions`, author `Pablo Santos`, today's date, and `\end{document}`. Compile the resulting `.tex` file with `pdflatex` (or on Overleaf).

The optional `standalone=False` produces the old table-fragment format.


In [6]:
def generate_tables(max_lam_degree, max_mu_degree, filename,
                    exploit_symmetry=False, include_zeros=False,
                    standalone=True, title=None, author=LATEX_AUTHOR):
    """Export all requested LaTeX tables to one complete, compilable document.

    To insert just table fragments into another TeX file, use standalone=False.
    """
    max_lam_degree, max_mu_degree = int(max_lam_degree), int(max_mu_degree)
    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    doc_title = title or LATEX_TITLE or f"Deg{max(max_lam_degree, max_mu_degree)} partitions"
    with path.open('w', encoding='utf-8') as f:
        if standalone:
            f.write(document_start(doc_title, author))
        for n in range(max_lam_degree + 1):
            for m in range(max_mu_degree + 1):
                if exploit_symmetry and n > m: continue
                for lam in partitions_of(n):
                    for mu in partitions_of(m):
                        if exploit_symmetry and n == m and lam > mu: continue
                        f.write(table_tex(lam, mu, include_zeros=include_zeros))
                        f.write("\n%--------------------\n")
        if standalone:
            f.write(document_end())
    return path

# Example:
output = generate_tables(9, , 'tex_tables/opt/deg9.tex')
# print(output)


## Performance notes and next steps

1. `coproduct` is cached, so all sparse LR decompositions are reused.
2. `kernel` is the expensive, but reusable, structural computation. In large batches, it can be precomputed once per degree and shared across target pairs.
3. Grouped coproduct data avoids generating impossible $(\lambda',\mu')$ candidates.
4. When only the **largest** nonzero degree is needed, `max_degree` is sufficient, but this is the largest degree appearing in the full coefficient sum; it is not automatically the **common single-partition expansion** maximum $M(\lambda,\mu)$ discussed earlier.
5. For an investigation of the **common single-partition** problem, a separate routine should compute $\mathcal E(\lambda,\mu)$ using one common $\delta$ rather than a pair $(\delta,\gamma)$.

**Dependencies:** SageMath, with `sage.libs.lrcalc.lrcalc.coprod`, `lrcoef_unsafe`, and `mult`. The code has not been executed in a SageMath runtime here; run the validation cell in your Sage environment before relying on generated tables.